# Data Preparation — UCSD Campus Species

Download images from iNaturalist, filter to the most common species, and split into `train/val/test` folders ready for `ImageFolder`.

## 1. Setup

In [ ]:
import pandas as pd
import requests
import os
import time
import random
import shutil
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from PIL import Image
from io import BytesIO

SEED = 42
random.seed(SEED)

### Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
CSV_PATH = Path("../observations-712098/observations-712098.csv")
DOWNLOAD_DIR = Path("images_raw")         # flat download cache (all images)
DATA_DIR = Path("data")                   # final train/val/test splits

# ── Filtering ──────────────────────────────────────────────────────────────────
TOP_N_SPECIES = 50                        # keep the N most common species
MIN_IMAGES_PER_CLASS = 15                 # drop species below this threshold

# ── Image quality ──────────────────────────────────────────────────────────────
IMAGE_SIZE_VARIANT = "large"              # iNaturalist size: square, small, medium, large, original

# ── Split ratios ───────────────────────────────────────────────────────────────
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

# ── Download ───────────────────────────────────────────────────────────────────
MAX_WORKERS = 8                           # parallel download threads
REQUEST_TIMEOUT = 15                      # seconds per request

print(f"CSV:              {CSV_PATH}")
print(f"Top N species:    {TOP_N_SPECIES}")
print(f"Min images/class: {MIN_IMAGES_PER_CLASS}")
print(f"Split ratios:     {TRAIN_RATIO}/{VAL_RATIO}/{TEST_RATIO}")
print(f"Image variant:    {IMAGE_SIZE_VARIANT}")

## 2. Load and Filter Observations

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Total observations: {len(df)}")
print(f"Unique species:     {df['scientific_name'].nunique()}")

# Drop rows without an image
df = df.dropna(subset=["image_url"]).copy()
print(f"With images:        {len(df)}")

# Species frequency
species_counts = df["scientific_name"].value_counts()

# Keep only top N species that also meet minimum threshold
top_species = species_counts.head(TOP_N_SPECIES)
top_species = top_species[top_species >= MIN_IMAGES_PER_CLASS]
selected_species = list(top_species.index)

df = df[df["scientific_name"].isin(selected_species)].copy()
df = df.reset_index(drop=True)

print(f"\nSelected {len(selected_species)} species, {len(df)} observations:")
for name in selected_species:
    common = df.loc[df["scientific_name"] == name, "common_name"].iloc[0]
    count = top_species[name]
    print(f"  {name:<45s} ({common}) — {count}")

## 3. Prepare Download URLs

Upgrade the iNaturalist image URLs from `medium` to the configured size variant.

In [ ]:
def upgrade_url(url: str, size: str = IMAGE_SIZE_VARIANT) -> str:
    """Replace the iNaturalist size token in the URL."""
    for old_size in ("square", "small", "medium", "large", "original"):
        if f"/{old_size}." in url:
            return url.replace(f"/{old_size}.", f"/{size}.")
    return url

# Build a clean label from scientific name (replace spaces with underscores)
df["label"] = df["scientific_name"].str.replace(" ", "_", regex=False)
df["download_url"] = df["image_url"].apply(upgrade_url)
df["filename"] = df["id"].astype(str) + ".jpg"

print(f"Sample URLs (before → after):")
for i in range(3):
    print(f"  {df['image_url'].iloc[i]}")
    print(f"  → {df['download_url'].iloc[i]}")
    print()

## 4. Download Images

Downloads are parallelized and cached — re-running this cell skips already-downloaded files.

In [ ]:
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

def download_one(row):
    """Download a single image. Returns (observation_id, success, error_msg)."""
    obs_id = row["id"]
    url = row["download_url"]
    dest = DOWNLOAD_DIR / row["filename"]

    if dest.exists():
        return (obs_id, True, "cached")

    try:
        resp = requests.get(url, timeout=REQUEST_TIMEOUT)
        resp.raise_for_status()

        # Validate it's actually an image
        img = Image.open(BytesIO(resp.content))
        img.verify()

        with open(dest, "wb") as f:
            f.write(resp.content)
        return (obs_id, True, "downloaded")
    except Exception as e:
        return (obs_id, False, str(e))


rows = df.to_dict("records")
results = []
failed = []

t0 = time.time()
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(download_one, row): row for row in rows}
    for i, future in enumerate(as_completed(futures), 1):
        obs_id, success, msg = future.result()
        results.append((obs_id, success, msg))
        if not success:
            failed.append((obs_id, msg))
        if i % 500 == 0 or i == len(rows):
            elapsed = time.time() - t0
            print(f"  [{i}/{len(rows)}] {elapsed:.0f}s elapsed, {len(failed)} failed")

elapsed = time.time() - t0
n_ok = sum(1 for _, s, _ in results if s)
print(f"\nDone in {elapsed:.1f}s — {n_ok} succeeded, {len(failed)} failed out of {len(rows)}")

if failed:
    print(f"\nFirst 10 failures:")
    for obs_id, msg in failed[:10]:
        print(f"  id={obs_id}: {msg}")

## 5. Remove Failed Downloads

Drop any observations whose images failed to download so they don't end up in the splits.

In [ ]:
failed_ids = {obs_id for obs_id, msg in failed}
before = len(df)
df = df[~df["id"].isin(failed_ids)].copy()
df = df.reset_index(drop=True)

# Also verify the file actually exists on disk (catches partial/corrupt downloads)
exists_mask = df["filename"].apply(lambda f: (DOWNLOAD_DIR / f).exists())
df = df[exists_mask].copy()
df = df.reset_index(drop=True)

print(f"Dropped {before - len(df)} observations with missing images")
print(f"Remaining: {len(df)} observations across {df['scientific_name'].nunique()} species")

# Re-check class counts after filtering
final_counts = df["scientific_name"].value_counts()
below_threshold = final_counts[final_counts < MIN_IMAGES_PER_CLASS]
if len(below_threshold) > 0:
    print(f"\nWarning: {len(below_threshold)} species now below {MIN_IMAGES_PER_CLASS} images after failed downloads:")
    for name, count in below_threshold.items():
        print(f"  {name}: {count}")
    df = df[~df["scientific_name"].isin(below_threshold.index)].copy()
    df = df.reset_index(drop=True)
    print(f"Dropped those. Now: {len(df)} observations, {df['scientific_name'].nunique()} species")
else:
    print("All species still meet the minimum threshold.")

## 6. Stratified Train / Val / Test Split

Split per-species so every class is represented proportionally in each split.

In [ ]:
def stratified_split(df, train_ratio, val_ratio, seed=42):
    """Split dataframe into train/val/test per species, preserving class proportions."""
    train_rows, val_rows, test_rows = [], [], []

    for species, group in df.groupby("scientific_name"):
        indices = group.index.tolist()
        random.seed(seed)
        random.shuffle(indices)

        n = len(indices)
        n_train = max(1, int(n * train_ratio))
        n_val = max(1, int(n * val_ratio))

        train_rows.extend(indices[:n_train])
        val_rows.extend(indices[n_train:n_train + n_val])
        test_rows.extend(indices[n_train + n_val:])

    return train_rows, val_rows, test_rows

train_idx, val_idx, test_idx = stratified_split(df, TRAIN_RATIO, VAL_RATIO, seed=SEED)

df["split"] = ""
df.loc[train_idx, "split"] = "train"
df.loc[val_idx, "split"] = "val"
df.loc[test_idx, "split"] = "test"

print(f"Train: {len(train_idx)}")
print(f"Val:   {len(val_idx)}")
print(f"Test:  {len(test_idx)}")
print(f"Total: {len(train_idx) + len(val_idx) + len(test_idx)}")

# Sanity check: every observation is assigned
assert df["split"].ne("").all(), "Some observations have no split assignment!"

## 7. Copy Images into `data/train/val/test` Folder Structure

Creates the `ImageFolder`-compatible layout expected by the training notebook.

In [ ]:
# Clean any previous split
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)

copied = 0
skipped = 0

for _, row in df.iterrows():
    split = row["split"]
    label = row["label"]
    src = DOWNLOAD_DIR / row["filename"]
    dest_dir = DATA_DIR / split / label
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / row["filename"]

    if src.exists():
        shutil.copy2(src, dest)
        copied += 1
    else:
        skipped += 1

print(f"Copied {copied} images into {DATA_DIR}/")
if skipped:
    print(f"Skipped {skipped} (source not found)")

## 8. Verify Final Dataset

In [ ]:
print(f"{' Split':<8} {'Classes':>8} {'Images':>8}")
print("-" * 28)

total_images = 0
for split in ["train", "val", "test"]:
    split_dir = DATA_DIR / split
    classes = sorted([d.name for d in split_dir.iterdir() if d.is_dir()])
    n_images = sum(len(list((split_dir / c).glob("*.jpg"))) for c in classes)
    total_images += n_images
    print(f"{split:<8} {len(classes):>8} {n_images:>8}")

print("-" * 28)
print(f"{'Total':<8} {'':>8} {total_images:>8}")

# Per-class breakdown
print(f"\nPer-class counts:")
print(f"{'Species':<45s} {'Train':>6} {'Val':>6} {'Test':>6} {'Total':>6}")
print("-" * 75)

all_classes = sorted([d.name for d in (DATA_DIR / "train").iterdir() if d.is_dir()])
for cls in all_classes:
    counts = {}
    for split in ["train", "val", "test"]:
        cls_dir = DATA_DIR / split / cls
        counts[split] = len(list(cls_dir.glob("*.jpg"))) if cls_dir.exists() else 0
    total = sum(counts.values())
    print(f"{cls:<45s} {counts['train']:>6} {counts['val']:>6} {counts['test']:>6} {total:>6}")

## 9. Validate Image Integrity

Attempt to open every image to catch truncated or corrupt files early.

In [ ]:
corrupt_files = []

for split in ["train", "val", "test"]:
    split_dir = DATA_DIR / split
    for img_path in sorted(split_dir.rglob("*.jpg")):
        try:
            with Image.open(img_path) as img:
                img.verify()
        except Exception as e:
            corrupt_files.append((str(img_path), str(e)))

if corrupt_files:
    print(f"Found {len(corrupt_files)} corrupt images:")
    for path, err in corrupt_files[:20]:
        print(f"  {path}: {err}")
    print("\nRemoving corrupt files...")
    for path, _ in corrupt_files:
        os.remove(path)
    print("Done. Re-run the verification cell to confirm.")
else:
    print(f"All images validated successfully.")

## Done

The `data/` directory is ready for the training notebook.

```
data/
  train/
    Species_one/
    Species_two/
    ...
  val/
    ...
  test/
    ...
```